In [ ]:
#instalare
#!pip install ultralytics

#yaml_content = """
# train: /content/dataset/images/train
# val: /content/dataset/images/val
# names: {0: 'face'}
# """
# with open("data.yaml", "w") as f:
#     f.write(yaml_content)

#antrenare
#from ultralytics import YOLO

# model = YOLO("yolov8n.pt")
# model.train(
#     data="data.yaml",
#     epochs=15,
#     imgsz=640,
#     batch=16,
#     project="runs/detect",
#     name="task1_yolo_colab",
#     mosaic=1.0
# )

In [ ]:
from ultralytics import YOLO
import glob
import os
import numpy as np
from tqdm import tqdm
import ntpath

TEST_IMAGE_SOURCE = "../../../../evaluare/fake_test"
SOLUTION_SAVE_DIRECTORY = "output"

detection_model = YOLO("best.pt")
print(f"starting run on {TEST_IMAGE_SOURCE}")

test_image_paths = sorted(glob.glob(os.path.join(TEST_IMAGE_SOURCE, "*.jpg")))

all_detection_boxes = []
all_confidence_scores = []
all_image_filenames = []

for image_path in tqdm(test_image_paths, desc="YOLO run"):
    inference_results = detection_model.predict(image_path, conf=0.001, iou=0.5, verbose=False)

    current_filename = ntpath.basename(image_path)
    for result in inference_results:

        boxes_xyxy = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()

        for box, score in zip(boxes_xyxy, scores):
            x1, y1, x2, y2 = map(int, box)
            all_detection_boxes.append([x1, y1, x2, y2])
            all_confidence_scores.append(float(score))
            all_image_filenames.append(current_filename)

os.makedirs(SOLUTION_SAVE_DIRECTORY, exist_ok=True)

np.save(os.path.join(SOLUTION_SAVE_DIRECTORY, 'detections_all_faces.npy'), np.array(all_detection_boxes))
np.save(os.path.join(SOLUTION_SAVE_DIRECTORY, 'scores_all_faces.npy'), np.array(all_confidence_scores))
np.save(os.path.join(SOLUTION_SAVE_DIRECTORY, 'file_names_all_faces.npy'), np.array(all_image_filenames))

print(f"saved {len(all_detection_boxes)} detections to {SOLUTION_SAVE_DIRECTORY}")